In [4]:
import pandas as pd

# Assuming 'df' is your DataFrame with the movie data
df = pd.read_csv('sample for algo testing_100K.csv')

# Create a user-movie matrix
user_movie_matrix = df.pivot(index='UserID', columns='MovieID', values='Rating')

# Fill NaN values with 0 (assuming NaN means the user hasn't rated the movie)
user_movie_matrix = user_movie_matrix.fillna(0)

# Display the user-movie matrix
print(user_movie_matrix)

MovieID  1     2     3     4     5     6     7     8     9     10    ...  \
UserID                                                               ...   
1         5.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
2         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
3         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
...       ...   ...   ...   ...   ...   ...   ...   ...   ...   ...  ...   
6036      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6037      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6038      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6039      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
6040      0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

MovieID  39

CD + LOD

In [2]:
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVDpp
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from surprise import accuracy

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Music Composer', 'Director', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Split the data into train and test sets
trainset, testset = train_test_split(data, test_size=0.25)

# Handle NaN values in the 'Music Composer,' 'Director,' and 'Starring' columns
df['Music Composer'].fillna('', inplace=True)
df['Director'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# One-hot encode 'Music Composer' and 'Director'
encoder_composer = OneHotEncoder(sparse=False)
composer_encoded = encoder_composer.fit_transform(df[['Music Composer']])

encoder_director = OneHotEncoder(sparse=False)
director_encoded = encoder_director.fit_transform(df[['Director']])

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(composer_encoded, columns=[f"Composer_{col}" for col in encoder_composer.get_feature_names_out(['Music Composer'])]),
    pd.DataFrame(director_encoded, columns=[f"Director_{col}" for col in encoder_director.get_feature_names_out(['Director'])]),
    pd.DataFrame(starring_encoded, columns=[f"Starring_{col}" for col in mlb_starring.classes_])
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Build the Trainset from the Surprise Dataset
trainset_with_features = data_with_features.build_full_trainset()

# Train the model using the extended user-movie matrix with features
model_with_features = SVDpp(n_factors=100, n_epochs=20)
model_with_features.fit(trainset_with_features)

# Make predictions on the test set
predictions_with_features = model_with_features.test(testset)

# Calculate RMSE, MAE, and MSE
rmse_with_features = accuracy.rmse(predictions_with_features)
mae_with_features = accuracy.mae(predictions_with_features)
mse_with_features = accuracy.mse(predictions_with_features)

print(f"RMSE with Composer, Director, and Starring: {rmse_with_features}")
print(f"MAE with Composer, Director, and Starring: {mae_with_features}")
print(f"MSE with Composer, Director, and Starring: {mse_with_features}")

C:\Users\Yui Chee Xuan\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\Yui Chee Xuan\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


RMSE: 0.5303
MAE:  0.4223
MSE: 0.2813
RMSE with Composer, Director, and Starring: 0.5303415440383603
MAE with Composer, Director, and Starring: 0.42232753783476873
MSE with Composer, Director, and Starring: 0.2812621533329921


In [5]:
from surprise.model_selection import GridSearchCV
from surprise import SVDpp
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer

# Assuming 'df' is your DataFrame with columns 'MovieID', 'Rating', 'Music Composer', 'Starring', and 'UserID'
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['UserID', 'MovieID', 'Rating']], reader)

# Handle NaN values in the 'Music Composer' and 'Starring' columns
df['Music Composer'].fillna('', inplace=True)
df['Starring'].fillna('', inplace=True)

# One-hot encode 'Music Composer'
encoder_composer = OneHotEncoder(sparse=False)
composer_encoded = encoder_composer.fit_transform(df[['Music Composer']])

# Convert 'Starring' to binary representation using MultiLabelBinarizer
mlb_starring = MultiLabelBinarizer()
starring_encoded = mlb_starring.fit_transform(df['Starring'].str.split(',').apply(lambda x: [i.strip() for i in x]))

# Combine the encoded features with the user-movie matrix
user_movie_matrix_with_features = pd.concat([
    df[['UserID', 'MovieID', 'Rating']],
    pd.DataFrame(composer_encoded, columns=[f"Composer_{col}" for col in encoder_composer.get_feature_names_out(['Music Composer'])]),
    pd.DataFrame(starring_encoded, columns=[f"Starring_{col}" for col in mlb_starring.classes_])
], axis=1)

# Convert the DataFrame to Surprise Dataset
data_with_features = Dataset.load_from_df(user_movie_matrix_with_features[['UserID', 'MovieID', 'Rating']], reader)

# Define the parameter grid for grid search
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [10, 20],
    'lr_all': [0.005, 0.01],
}

# Create an SVD++ instance
algo = SVDpp()

# Create GridSearchCV instance
gs = GridSearchCV(algo_class=SVDpp, param_grid=param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data_with_features)

# Get the best parameters and results
best_params = gs.best_params['rmse']
best_rmse = gs.best_score['rmse']

# Display the best parameters and results
print(f"Best Parameters: {best_params}")
print(f"Best RMSE: {best_rmse}")

# Display the results for all combinations
results_df = pd.DataFrame.from_dict(gs.cv_results)
print(results_df[['params', 'mean_test_rmse', 'mean_test_mae']])

C:\Users\Yui Chee Xuan\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Best Parameters: {'n_factors': 50, 'n_epochs': 10, 'lr_all': 0.01}
Best RMSE: 0.9562839264980948
                                              params  mean_test_rmse  \
0  {'n_factors': 50, 'n_epochs': 10, 'lr_all': 0....        0.965477   
1  {'n_factors': 50, 'n_epochs': 10, 'lr_all': 0.01}        0.956284   
2  {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0....        0.956929   
3  {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.01}        0.971090   
4  {'n_factors': 100, 'n_epochs': 10, 'lr_all': 0...        0.970241   
5  {'n_factors': 100, 'n_epochs': 10, 'lr_all': 0...        0.963170   
6  {'n_factors': 100, 'n_epochs': 20, 'lr_all': 0...        0.964001   
7  {'n_factors': 100, 'n_epochs': 20, 'lr_all': 0...        0.975081   

   mean_test_mae  
0       0.770852  
1       0.760356  
2       0.761303  
3       0.770054  
4       0.775115  
5       0.766782  
6       0.767637  
7       0.775298  
